# Reversible Computing and Uncomputation ("Garbage Take-Out")
**Notebook:** Demonstrates why quantum circuits must clean up ancilla qubits to avoid entanglement artifacts

## Overview

In quantum computing, intermediate ("ancilla") qubits used during a computation must be
**uncomputed** (reset to $|0\rangle$) before being discarded. Failing to do so leaves them
entangled with the output, corrupting results â€” this leftover entanglement is called **garbage**.

### The problem
Suppose we compute $f(x)$ using an ancilla $|a\rangle$:
$$|x\rangle|0\rangle \xrightarrow{\text{compute}} |x\rangle|f(x)\rangle$$
If we discard the ancilla here, tracing it out introduces **mixed states** in the output.

### The solution: uncomputation
Run the computation forward, copy the result to a dedicated output register, then
run the computation **backwards** (apply all gates in reverse) to restore the ancilla:
$$|x\rangle|0\rangle|0\rangle \to |x\rangle|f(x)\rangle|0\rangle^{\text{ancilla restored}}$$

This keeps the total state **pure** and the ancilla **separable** (disentangled).

### Why quantum gates can be reversed
All quantum gates are **unitary** ($U^\dagger U = I$), so every gate has an inverse.
Running a circuit in reverse undoes the computation exactly.

In [ ]:
import qiskit as qk
from qiskit import QuantumCircuit, QuantumRegister, transpile
from qiskit.quantum_info import Statevector, Operator, partial_trace
from qiskit.visualization import plot_bloch_multivector
from math import pi
import numpy as np

import qiskit as qk
from qiskit import QuantumCircuit, QuantumRegister, transpile
from qiskit.quantum_info import Statevector, Operator, partial_trace
from qiskit.visualization import plot_bloch_multivector
from math import pi
import numpy as np

In [ ]:
# --- Target operation U_f ---
# U_f = CNOT(input -> output): the ideal function we want to compute.
# The ancilla qubit is present in the register but left untouched here,
# showing what the circuit would look like if we had direct access to U_f.
input_bit = QuantumRegister(1, 'input')
output_bit = QuantumRegister(1, 'output')
garbage_bit = QuantumRegister(1, 'ancilla')

Uf = QuantumCircuit(input_bit, output_bit, garbage_bit)
Uf.cx(input_bit[0], output_bit[0])  # implements f(x) = x (identity/copy) via CNOT

Uf.draw(output='mpl',justify='none')

In [ ]:
# --- Copy gate ---
# CNOT(output -> final-output): fans out the result of U_f into a
# dedicated 'final-output' register so we can later uncompute output
# back to |0> while preserving the answer in final-output.
final_output_bit = QuantumRegister(1, 'final-output')

copy = QuantumCircuit(output_bit, final_output_bit)
copy.cx(output_bit, final_output_bit)  # copy: |a>|0> -> |a>|a>
#copy.barrier()
copy.draw(output='mpl',justify='none')

In [ ]:
# --- Noisy/implementable operation V_f ---
# V_f = CNOT(input -> ancilla) then CNOT(input -> output).
# This is the circuit we CAN build with available gates, but it
# entangles the ancilla with the output â€” the "garbage" problem.
# The ancilla ends up in state |f(x)> = |x>, which is NOT |0>.
Vf = QuantumCircuit(input_bit, output_bit, garbage_bit)
Vf.barrier()
Vf.cx(input_bit[0], garbage_bit[0])  # step 1: copy input into ancilla (creates garbage)
Vf.cx(input_bit[0], output_bit[0])   # step 2: copy input into output (the desired result)
Vf.draw(output='mpl',justify='none')

In [ ]:
# --- Full uncomputation circuit: V_fâ€  âˆ˜ copy âˆ˜ V_f ---
# The sequence is:
#   1. Apply V_f forward:      |x>|0>|0> -> |x>|f(x)>|f(x)>  (ancilla = garbage)
#   2. Apply copy gate:        copies output into final-output register
#   3. Apply V_fâ€   (inverse):  undoes step 1, restoring ancilla to |0>
# Net effect: |x>|0>|0>|0> -> |x>|0>|0>|f(x)>  â€” ancilla is clean, result preserved.
Vf.inverse().compose(copy).compose(Vf).draw(output='mpl', justify='none')

In [ ]:
# Verify that V_fâ€  âˆ˜ copy âˆ˜ V_f acts the same as U_f on the output qubit
# and that the ancilla is restored to |0âŸ© (uncomputation works).

from qiskit.quantum_info import Statevector, partial_trace
import numpy as np

# Build V_fâ€  âˆ˜ copy âˆ˜ V_f on 4 qubits: input(0), output(1), ancilla(2), final-output(3)
input_bit     = QuantumRegister(1, 'input')
output_bit    = QuantumRegister(1, 'output')
garbage_bit   = QuantumRegister(1, 'ancilla')
final_out_bit = QuantumRegister(1, 'final-output')

# U_f: just CNOT(input â†’ output), ancilla and final-output untouched
# This is the reference/ideal circuit we want to emulate
Uf_full = QuantumCircuit(input_bit, output_bit, garbage_bit, final_out_bit)
Uf_full.cx(input_bit[0], output_bit[0])

# V_f: CNOT(input â†’ ancilla) then CNOT(input â†’ output)
# This is the imperfect gate that creates garbage in the ancilla
Vf_full = QuantumCircuit(input_bit, output_bit, garbage_bit, final_out_bit)
Vf_full.cx(input_bit[0], garbage_bit[0])
Vf_full.cx(input_bit[0], output_bit[0])

# copy: CNOT(output â†’ final-output)
# Saves the computed result before we uncompute V_f
copy_full = QuantumCircuit(input_bit, output_bit, garbage_bit, final_out_bit)
copy_full.cx(output_bit[0], final_out_bit[0])

# Full uncomputation circuit: V_fâ€  âˆ˜ copy âˆ˜ V_f
# compose() chains circuits left-to-right; .inverse() reverses all gates
combined = Vf_full.inverse().compose(copy_full).compose(Vf_full)
combined.draw(output='mpl', justify='none')

In [ ]:
# Numerical verification: check ancilla is restored and output matches U_f
# Tests all four computational basis input states (input x ancilla combinations)
test_labels = ['|0000âŸ©', '|0010âŸ©', '|1000âŸ©', '|1010âŸ©']
test_inputs  = ['0000',   '0010',   '1000',   '1010']

print("Testing uncomputation: V_fâ€  âˆ˜ copy âˆ˜ V_f")
print(f"{'Input':<12} {'Ancilla restored?':<22} {'Output matches U_f?'}")
print("-" * 55)
for label, inp in zip(test_labels, test_inputs):
    # Evolve each basis state through both circuits for comparison
    sv_ref  = Statevector.from_label(inp).evolve(Uf_full)   # reference: ideal U_f
    sv_comb = Statevector.from_label(inp).evolve(combined)  # test: uncomputation circuit

    # Check ancilla (qubit index 2) is back to |0âŸ© after uncomputation.
    # partial_trace traces out all qubits except the ancilla, giving its reduced density matrix.
    # A pure |0><0| state has density matrix diag(1, 0).
    rho_ancilla = partial_trace(sv_comb, [0, 1, 3])
    ancilla_ok = np.allclose(rho_ancilla.data, [[1, 0], [0, 0]], atol=1e-9)

    # Check output qubit (index 1) marginal matches U_f output.
    # Both reduced density matrices should be identical if uncomputation is correct.
    rho_out_ref  = partial_trace(sv_ref,  [0, 2, 3])
    rho_out_comb = partial_trace(sv_comb, [0, 2, 3])
    output_ok = np.allclose(rho_out_ref.data, rho_out_comb.data, atol=1e-9)

    print(f"{label:<12} {'âœ“' if ancilla_ok else 'âœ—':<22} {'âœ“' if output_ok else 'âœ—'}")

print("\nUncomputation verified: ancilla restored, output correct!")